# Wafer Guard · Repaired Fab-7 Excursion Detector

This notebook is the **audited and repaired** version of `original/bnypi.ipynb`.

**How to read it.** Every section answers one question, in this order:

| § | Question | Phase |
|---|---|---|
| 1 | What does the data actually contain? | Understand |
| 2 | Does the original really score F1 ≥ 0.90? | Reproduce |
| 3 | *Why* does it score that high? (evidence for every defect) | Audit |
| 4 | How do we build a pipeline that respects time and tensor shapes? | Repair |
| 5 | What is the honest performance at natural defect prevalence? | Re-evaluate |
| 6 | Does that hold up across time (walk-forward)? | Re-evaluate |
| 7 | How much of the fake 0.90 does each bug explain? | Attribute |
| 8 | One table: original vs repaired vs baselines | Summarise |
| 9 | Save artifacts + every measured number to `results/metrics.json` | Ship |

**Rule followed throughout:** no number in this project is typed by hand.
Every metric is computed in a cell below and written to `results/metrics.json`;
`AUDIT.md` and the Release Board app read their numbers from that file.

## 0 · Setup

Same libraries and **the same hyper-parameters as the original** (`HP` is copied unchanged).
We deliberately do *not* tune the model: the goal is to fix defects, so any change in the
score can be attributed to a fix, not to tuning.

In [1]:
import json
import random
import warnings
from dataclasses import dataclass, replace
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             matthews_corrcoef, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 1337                                   # same seed as the original notebook
EVAL_SEEDS = [1337, 7, 42, 2024, 31337]       # extra seeds, to show run-to-run variation


def set_seed(seed: int) -> None:
    """Make numpy, python and torch random numbers repeatable."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Unchanged from the original notebook.
HP = dict(batch_size=64, epochs=40, lr=1e-3, weight_decay=1e-5,
          hidden=256, latent=32, lambda_clf=1.0, test_size=0.20)

# New settings introduced by the repair (chosen up-front, never tuned on the test set).
SPLIT_FRACTIONS = dict(train=0.60, val=0.20)  # test = the most recent 20 % of wafers
CLIP_Z = 5.0          # cap standardised values at ±5σ (see §4.2 for why)
LEGACY_CODES = (100.0, 200.0, 300.0)          # PS-L firmware error codes (from the brief)
GATE_F1 = 0.90        # the release gate used by the original notebook

DATA_PATH = Path("data/secom_telemetry_modified.csv")
RESULTS_DIR = Path("results")
FIG_DIR = RESULTS_DIR / "figures"
ARTIFACT_DIR = Path("artifacts")
for d in (RESULTS_DIR, FIG_DIR, ARTIFACT_DIR):
    d.mkdir(parents=True, exist_ok=True)

RESULTS: dict = {}    # every measured number is collected here and saved in §9
print(f"torch {torch.__version__} | device={DEVICE}")

torch 2.10.0+cpu | device=cpu


Chart styling (one shared look for every figure: thin marks, quiet grid, a fixed colour order).

In [2]:
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
BLUES = ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "sans-serif", "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titleweight": "semibold", "axes.titlesize": 11, "axes.titlelocation": "left",
    "axes.edgecolor": AXIS, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False,
    "lines.linewidth": 2, "lines.solid_capstyle": "round",
})


def save_fig(fig, name: str) -> None:
    """Save a figure to results/figures/<name>.png and show it."""
    fig.savefig(FIG_DIR / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)